# 🛢️ Oil Region Profitability Analysis 🗼

**Objective:** Develop a machine learning model to identify the most profitable region for opening 200 new oil wells based on predicted oil reserves and risk analysis.

The project aims to:
- Predict the reserve volume of new oil wells.
- Select the top-performing wells from each region.
- Estimate potential profits under a fixed development budget.
- Evaluate financial risk using bootstrapping simulations.
- Choose the region with the highest expected profit and acceptable risk level.

**Project Tasks:**
1. Train a Linear Regression model to predict oil reserve volumes for new wells.
2. Select the 200 wells with the highest predicted reserves from 500 sampled locations.
3. Calculate expected profits considering.
4. Perform risk assessment using bootstrapping.
5. Identify the region with the highest average profit and a risk of loss below 2.5%.

**Problem Type:**
- Linear Regression
- Confidence Interval
- Bootstrapping

**Evaluation Metrics:**
- RMSE (Root Mean Squared Error)
- Total Profit
- Average Profit
- Risk of Loss

## 1. Data Loading and Preparation 📥

### 1.1 Load and Inspect Data 📥🔎

In [ ]:
#Import Libraries
import pandas as pd
import numpy as np

In [ ]:
#Load dataset into a DataFrame
geodata0 = pd.read_csv('C:/Users/Acer/Downloads/geo_data_0.csv')
geodata1 = pd.read_csv('C:/Users/Acer/Downloads/geo_data_1.csv')
geodata2 = pd.read_csv('C:/Users/Acer/Downloads/geo_data_2.csv') 

In [ ]:
#Inspect Data
geodata0.info()
print()
geodata1.info()
print()
geodata2.info()

***No missing values were detected in any of the datasets.***

In [ ]:
print(geodata0.sample(3))
print()
print(geodata1.sample(3))
print()
print(geodata2.sample(3))

In [ ]:
#Inspect Basic Correlation
geodata0[['f0', 'f1', 'f2', 'product']].corr()

**Key Correlations**
1. f0 vs f1 → -0.44 (moderately negative)
A moderate inverse relationship. When f0 increases, f1 tends to decrease.
It is not extremely strong, but it is significant.

2. f2 vs product → 0.48 (moderately positive)
This is the most important relationship in the entire matrix.

***The most important variable is f2. It has the highest correlation with `product` (0.48) and shows weak linear relationships with the other features.***

### 1.2 Data Preprocessing 📋

#### 1.2.1 Check for duplicate values

In [ ]:
geodata0.duplicated().sum()

In [ ]:
geodata1.duplicated().sum()

In [ ]:
geodata2.duplicated().sum()

***There are no duplicate values.***

In [ ]:
#Check for duplicate wells (id column)
geodata0[geodata0['id'].duplicated(keep=False)].sort_values('id')

In [ ]:
geodata1[geodata1['id'].duplicated(keep=False)].sort_values('id')

In [ ]:
geodata2[geodata2['id'].duplicated(keep=False)].sort_values('id')

***The fact that there are repeating IDs, but with different features, suggests that each row is likely a distinct measurement or estimate of the same well, rather than necessarily a duplication error.***

The `id` column will not be used for model training because it is a unique identifier and does not provide predictive geological information.

#### 1.2.2 Define Features and Target

In [ ]:
#Region 0
features0 = geodata0.drop(['id', 'product'], axis=1)
target0 = geodata0['product']

#Region 1
features1 = geodata1.drop(['id', 'product'], axis=1)
target1 = geodata1['product']

#Region 2
features2 = geodata2.drop(['id', 'product'], axis=1)
target2 = geodata2['product']

***Important:***

Feature scaling was not required for this project because Linear Regression is not sensitive to feature magnitudes in the same way as distance-based algorithms.

Additionally, the feature ranges appeared reasonably comparable, and model performance was satisfactory without normalization.

## 2. Model Training 🦾

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def train_and_evaluate_model(features, target):
    #Split data (75:25)
    features_train, features_valid, target_train, target_valid = train_test_split(
        features, target, test_size=0.25, random_state=12345)

    #Train model
    model = LinearRegression()
    model.fit(features_train, target_train)

    #Predict on validation set
    predictions = model.predict(features_valid)

    #Calculate RMSE
    rmse = mean_squared_error(target_valid, predictions) ** 0.5

    #Average predicted volume (useful for later profit analysis)
    avg_prediction = predictions.mean()

    return predictions, target_valid, rmse, avg_prediction

In [ ]:
#Run function for the 3 regions.
preds0, valid0, rmse0, avg0 = train_and_evaluate_model(features0, target0)
preds1, valid1, rmse1, avg1 = train_and_evaluate_model(features1, target1)
preds2, valid2, rmse2, avg2 = train_and_evaluate_model(features2, target2)

In [ ]:
#Show Results
print("Region 0")
print("RMSE:", rmse0)
print("Average predicted reserves:", avg0)
print()

print("Region 1")
print("RMSE:", rmse1)
print("Average predicted reserves:", avg1)
print()

print("Region 2")
print("RMSE:", rmse2)
print("Average predicted reserves:", avg2)

In [ ]:
#EXTRA
print(target0.describe())
print(target1.describe())
print(target2.describe())

For Region 0, the RMSE is 37.58, with an estimated average reserve of 92.59 thousand barrels. The well with the highest estimated reserve contains 185.36 thousand barrels.

For Region 1, the RMSE is the lowest at 0.89, indicating highly accurate predictions. However, this region also has the lowest estimated average reserve, at 68.73 thousand barrels. The well with the highest reserve contains 137.95 thousand barrels.

For Region 2, the RMSE is the highest at 40.03, with an estimated average reserve of 94.96 thousand barrels. The well with the largest reserve contains 190.03 thousand barrels.

Overall, while Region 1 provides the most accurate predictions, its estimated reserve volumes are considerably lower than those of Regions 0 and 2, which show higher reserve estimates but also substantially larger prediction errors.

## 3. Preparation for Profit Calculation 📊

**Key Values**

200 wells

Budget = $100M USD

Revenue per unit = $4,500 USD
Product = Thousands of barrels

In [ ]:
#Store constants
budget = 100_000_000      #100 million dollars
wells = 200               #Number of wells to develop
revenue_per_unit = 4500   #Revenue per thousand barrels

In [ ]:
#Minimum reserves required per well to avoid losses
break_even_volume = budget / wells / revenue_per_unit

print("Break-even volume per well:", break_even_volume)

***This means that each well needs to produce at least 111.1 thousand barrels to recover the investment.***

In [ ]:
#Compare with average reserves
print("Break-even volume:", break_even_volume)
print()

print("Region 0 average reserves:", avg0)
print("Region 1 average reserves:", avg1)
print("Region 2 average reserves:", avg2)

No single region reaches the minimum required average volume.

Developing all wells indiscriminately would likely not be profitable.

Region 1 stands out for its extremely low RMSE, indicating highly accurate predictions, although its average reserves are significantly lower than the other regions.

***How to prepare the step for calculating profit.***

The average predicted reserves in all three regions are below the threshold (111.11), which suggests that developing wells without selection would likely lead to losses.

However, the company’s strategy is to select only the top-performing wells based on predicted reserves. Therefore, the next stage will focus on identifying the 200 wells with the highest predicted production and calculating the expected profit using their actual reserve values.

Additionally, the prediction accuracy of each model will play an important role in the reliability of the profit estimates, especially considering that Region 1 showed significantly lower RMSE than the other regions.

## 4. Calculation of the Profit from a Set of selected Oil Wells and Modeling of Predictions 📈💰

***Tasks:***

- Take predictions and actual values
- Select the 200 wells with the highest predictions
- Obtain the actual reserves for those wells
- Calculate revenue
- Subtract investment

In [ ]:
#Create Profit Function
def calculate_profit(predictions, target, count):
    #Convert to Series and reset indexes
    predictions = pd.Series(predictions).reset_index(drop=True)
    target = target.reset_index(drop=True)

    #Select top wells with highest predictions
    top_wells = predictions.sort_values(ascending=False).head(count)

    #Get actual reserves for selected wells
    selected_reserves = target[top_wells.index]

    #Calculate total reserves
    total_reserves = selected_reserves.sum()

    #Calculate revenue
    revenue = total_reserves * revenue_per_unit

    #Calculate profit
    profit = revenue - budget

    return profit, total_reserves

In [ ]:
#Run Profit Function for the 3 regions
profit0, reserves0 = calculate_profit(preds0, valid0, 200)
profit1, reserves1 = calculate_profit(preds1, valid1, 200)
profit2, reserves2 = calculate_profit(preds2, valid2, 200)

print("Region 0")
print("Total reserves:", reserves0)
print("Profit:", profit0)
print()

print("Region 1")
print("Total reserves:", reserves1)
print("Profit:", profit1)
print()

print("Region 2")
print("Total reserves:", reserves2)
print("Profit:", profit2)

Although Region 1 achieved the lowest RMSE, indicating highly reliable predictions, Region 0 generated the highest estimated profit after selecting the top 200 wells.

This suggests that higher average reserves in Region 0 compensated for its larger prediction error.

At this stage, Region 0 appears to be the most profitable candidate for development. However, a final decision should also consider financial risk and uncertainty through bootstrapping analysis.

## 5. Calculation of Risks and Profits for Each Region (Bootstrapping) 🛡️💰💸

In [ ]:
#Create Bootstrap function
def bootstrap_profit(target, predictions):
    predictions = pd.Series(predictions).reset_index(drop=True)
    target = pd.Series(target).reset_index(drop=True)

    state = np.random.RandomState(12345)

    profits = []

    for i in range(1000):
        target_subsample = target.sample(n=500, replace=True, random_state=state)

        predictions_subsample = predictions.iloc[target_subsample.index]

        profit, _ = calculate_profit(predictions_subsample, target_subsample, 200)

        profits.append(profit)

    profits = pd.Series(profits)

    mean_profit = profits.mean()

    lower = profits.quantile(0.025)
    upper = profits.quantile(0.975)

    risk = (profits < 0).mean() * 100

    return profits, mean_profit, lower, upper, risk

In [ ]:
#Run Bootstrap Function for the 3 Regions
profits0, mean0, low0, up0, risk0 = bootstrap_profit(valid0, preds0)
profits1, mean1, low1, up1, risk1 = bootstrap_profit(valid1, preds1)
profits2, mean2, low2, up2, risk2 = bootstrap_profit(valid2, preds2)

In [ ]:
#Show Results
print("Region 0")
print(profits0.head())
print()

print("Region 1")
print(profits1.head())
print()

print("Region 2")
print(profits2.head())

In [ ]:
#Show Distribution (Histogram)
profits0.hist(bins=50)

In [ ]:
profits1.hist(bins=50)

In [ ]:
profits2.hist(bins=50)

Bootstrapping with 1000 samples was applied to estimate the distribution of profits for each region.

For every iteration:

* 500 wells were sampled with replacement.
* The top 200 wells were selected based on predicted reserves.
* Profit was calculated using the actual reserves of the selected wells.

This simulation approach allows estimating not only expected profit, but also the uncertainty and financial risk associated with each region.

## Summary Table 📌

In [ ]:
#Save results in a DataFrame (Summary Table)
bootstrap_results = pd.DataFrame({
    'Region': ['0', '1', '2'],
    'Average Profit': [mean0, mean1, mean2],
    '95% CI Lower': [low0, low1, low2],
    '95% CI Upper': [up0, up1, up2],
    'Risk of Loss (%)': [risk0, risk1, risk2]})

In [ ]:
#Convert to Millions
money_columns = ['Average Profit', '95% CI Lower', '95% CI Upper']

bootstrap_results[money_columns] = (bootstrap_results[money_columns] / 1_000_000)

#Rename Columns
bootstrap_results.rename(columns={
    'Average Profit': 'Average Profit (Million USD)',
    '95% CI Lower': '95% CI Lower (Million USD)',
    '95% CI Upper': '95% CI Upper (Million USD)'
}, inplace=True)

In [ ]:
print(bootstrap_results)

The bootstrapping analysis provided estimates of the average profit, confidence intervals, and probability of financial loss for each region.

The risk of loss was calculated as the percentage of bootstrap samples with negative profit values.

Regions with a risk higher than 2.5% do not satisfy the requirements and should not be considered for development.

## Final Conclusion 🏁

Although Region 0 showed the highest estimated profit during the initial profit calculation, the bootstrapping analysis revealed a higher probability of financial loss.

**Region 1** demonstrated the lowest risk of loss while maintaining stable and positive average profits. Additionally, its prediction model achieved the lowest RMSE, indicating highly reliable reserve estimates.

Since one requirement was to select a region with a loss risk below 2.5%, **Region 1 is the only candidate for oil well development.** The confidence interval for Region 1 remained mostly positive, reinforcing the stability and reliability of its projected profits compared to the other regions.